# 01 - Introduction to Residue Manifold Learning (RML) & CGCS

**Grok Track**  
**Allen Lab Collaboration**  
**Trisomy 21 Project**

Introduction to the core framework.

This notebook is designed to run even if some `src/grok/` files are not ready yet.

It will:

```text
try src/grok/
→ if import works, use repo functions
→ if import fails, use notebook fallback functions
→ run the demo either way
```


In [ ]:
# ================================================
# SETUP: Colab + local
# ================================================
from pathlib import Path
import sys
import subprocess
import math
import numpy as np
import matplotlib.pyplot as plt

REPO_NAME = "allen-lab-report-tool"
REPO_URL = "https://github.com/thinkthoughts/allen-lab-report-tool.git"

cwd = Path.cwd()

if (cwd / "src").exists():
    repo_root = cwd
elif cwd.name == "grok" and cwd.parent.name == "notebooks":
    repo_root = cwd.parents[1]
elif (cwd / REPO_NAME).exists():
    repo_root = cwd / REPO_NAME
else:
    print("Repo not found in current runtime. Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    repo_root = cwd / REPO_NAME

src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("cwd:", cwd)
print("repo_root:", repo_root)
print("src_path:", src_path)
print("src exists:", src_path.exists())


## 1. Load Grok Package or Use Fallbacks

In [ ]:
USING_GROK_SRC = False

try:
    import grok
    from grok import *
    from grok.cgcs import calculate_cgcs, calculate_cgcs_trisomy
    from grok.trisomy_metrics import trisomy_cgcs_score, simulate_intervention_recovery
    from grok.visualization import plot_dosage_impact, plot_cgcs_vs_noise

    USING_GROK_SRC = True
    print("🎉 Grok RML package loaded successfully!")
    print(f"Version: {getattr(grok, '__version__', 'unknown')}")

except Exception as e:
    print("Grok src import failed. Using notebook fallback functions.")
    print("Import error:", repr(e))

    def calculate_cgcs(*components):
        """Multiplicative coherence score over normalized components."""
        if not components:
            return 0.0
        clipped = [max(0.0, min(1.0, float(c))) for c in components]
        product = 1.0
        for c in clipped:
            product *= c
        return product ** (1.0 / len(clipped))

    def calculate_cgcs_trisomy(
        dosage_ratio=1.5,
        transcription_weight=1.0,
        propagation_weight=0.9,
        variability_weight=0.85,
        compensation_weight=0.8,
        return_components=False,
    ):
        """Toy CGCS model for dosage perturbation."""
        dosage_noise = abs(float(dosage_ratio) - 1.0)

        transcription = max(0.0, 1.0 - transcription_weight * dosage_noise)
        propagation = max(0.0, 1.0 - propagation_weight * dosage_noise)
        variability = max(0.0, 1.0 - variability_weight * dosage_noise)
        compensation = max(0.0, 1.0 - compensation_weight * dosage_noise)

        cgcs = calculate_cgcs(
            transcription,
            propagation,
            variability,
            compensation,
        )

        if return_components:
            return {
                "dosage_ratio": dosage_ratio,
                "dosage_noise": dosage_noise,
                "transcription": transcription,
                "propagation": propagation,
                "variability": variability,
                "compensation": compensation,
                "cgcs": cgcs,
            }

        return cgcs

    def trisomy_cgcs_score(dosage_ratio=1.5, return_components=False):
        return calculate_cgcs_trisomy(
            dosage_ratio=dosage_ratio,
            return_components=return_components,
        )

    def simulate_intervention_recovery(baseline_cgcs, recovery_strength=0.45):
        """Project baseline coherence upward toward 1.0 by recovery strength."""
        baseline_cgcs = float(baseline_cgcs)
        recovery_strength = max(0.0, min(1.0, float(recovery_strength)))
        return baseline_cgcs + recovery_strength * (1.0 - baseline_cgcs)

    def plot_dosage_impact():
        ratios = np.linspace(1.0, 1.8, 81)
        scores = [trisomy_cgcs_score(r) for r in ratios]

        plt.figure(figsize=(7, 4))
        plt.plot(ratios, scores, linewidth=2)
        plt.axvline(1.0, linestyle="--", linewidth=1)
        plt.axvline(1.5, linestyle="--", linewidth=1)
        plt.xlabel("Dosage ratio")
        plt.ylabel("CGCS")
        plt.title("CGCS decreases under dosage perturbation")
        plt.grid(True, alpha=0.3)
        plt.show()

    def plot_cgcs_vs_noise():
        noise = np.linspace(0.0, 0.8, 81)
        ratios = 1.0 + noise
        scores = [trisomy_cgcs_score(r) for r in ratios]

        plt.figure(figsize=(7, 4))
        plt.plot(noise, scores, linewidth=2)
        plt.xlabel("Dosage noise = |ratio - 1.0|")
        plt.ylabel("CGCS")
        plt.title("CGCS vs dosage noise")
        plt.grid(True, alpha=0.3)
        plt.show()

print("Using src/grok functions:", USING_GROK_SRC)


## 2. Core CGCS

CGCS is used here as a compact multiplicative coherence score.

```text
perfect component alignment → CGCS near 1.0
dosage perturbation/noise   → lower CGCS
recovery/projection         → CGCS moves upward again
```


In [ ]:
perfect = calculate_cgcs(1.0, 1.0, 1.0, 1.0, 1.0)
print(f"Perfect structure CGCS: {perfect:.4f}")


## 3. Trisomy 21 Dosage Noise Example

Simple dosage comparison:

```text
normal dosage  = 1.0x
trisomy dosage = 1.5x
```


In [ ]:
normal = trisomy_cgcs_score(dosage_ratio=1.0, return_components=True)
trisomy = trisomy_cgcs_score(dosage_ratio=1.5, return_components=True)

print(f"Normal (1.0x dosage)  → CGCS: {normal['cgcs']:.4f}")
print(f"Trisomy 21 (1.5x)     → CGCS: {trisomy['cgcs']:.4f}")

print("\nNormal components:")
for k, v in normal.items():
    print(f"  {k}: {v}")

print("\nTrisomy components:")
for k, v in trisomy.items():
    print(f"  {k}: {v}")


## 4. Dosage Impact Visualization

In [ ]:
plot_dosage_impact()


## 5. CGCS vs Noise Visualization

In [ ]:
plot_cgcs_vs_noise()


## 6. Intervention Recovery Simulation

In [ ]:
baseline = trisomy["cgcs"]
recovered = simulate_intervention_recovery(baseline, recovery_strength=0.45)

print(f"Baseline CGCS (Trisomy 21): {baseline:.4f}")
print(f"After simulated intervention: {recovered:.4f}")
print(f"Delta recovery: {recovered - baseline:.4f}")


## 7. Key Ideas

- **RML**: Studies structure on modular manifolds.
- **CGCS**: Multiplicative score of coherence under constraints.
- **Trisomy 21**: Modeled here as dosage noise that lowers CGCS.
- **Recovery**: Therapies and drugs can be modeled as projection/recovery steps that raise CGCS.

**Next**: Notebook 02 — Full Trisomy 21 Prototype.
